# CXR-LT 2023

This notebook is the merged CXR-LT 2023 examination pass. It combines the original split/schema checks with the follow-on study, timeline, co-occurrence, view-conditioned, image-path, and report-linkage checks.

It reports both split-level summaries and global summaries that ignore the train/development/test split. Bar charts annotate each bar with both count and percentage: vertical bars show labels above the bar, and horizontal bars show labels at the bar end.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

cwd = Path.cwd()
if cwd.name != "00-examine-data":
    raise Exception("Please run this notebook from the 00-examine-data directory")

root_dir = cwd.parents[1]
data_dir = root_dir / "data"
cxr_lt_2023_dir = data_dir / "CXR-LT" / "cxr-lt-multi-label-long-tailed-classification-on-chest-x-rays-2.0.0" / "cxr-lt-2023"
mimic_cxr_dir = data_dir / "MIMIC-CXR"
mimic_cxr_jpg_dir = data_dir / "MIMIC-CXR-JPG"

print(f"root_dir: {root_dir}")
print(f"cxr_lt_2023_dir: {cxr_lt_2023_dir}")
print(f"mimic_cxr_dir exists: {mimic_cxr_dir.exists()}")
print(f"mimic_cxr_jpg_dir exists: {mimic_cxr_jpg_dir.exists()}")


In [ ]:
CSV_FILES = {
    "train": "train.csv",
    "development": "development.csv",
    "test": "test.csv",
    "development_sample_submission": "development_sample_submission.csv",
    "test_sample_submission": "test_sample_submission.csv",
}

ANALYSIS_SPLIT_NAMES = ["train", "development", "test"]

ID_COLUMNS = [
    "dicom_id",
    "subject_id",
    "study_id",
    "ViewPosition",
    "ViewCodeSequence_CodeMeaning",
    "path",
]


def load_dataset(filename: str) -> pd.DataFrame:
    return pd.read_csv(cxr_lt_2023_dir / filename)


def load_first_existing(paths: list[Path]) -> tuple[pd.DataFrame | None, Path | None]:
    for candidate_path in paths:
        if candidate_path.exists():
            return pd.read_csv(candidate_path), candidate_path
    return None, None


datasets = {name: load_dataset(filename) for name, filename in CSV_FILES.items()}

train_df = datasets["train"]
dev_df = datasets["development"]
test_df = datasets["test"]
dev_sample_sub_df = datasets["development_sample_submission"]
test_sample_sub_df = datasets["test_sample_submission"]

analysis_splits = {split_name: datasets[split_name] for split_name in ANALYSIS_SPLIT_NAMES}
submission_splits = {
    "development_sample_submission": dev_sample_sub_df,
    "test_sample_submission": test_sample_sub_df,
}

global_df = pd.concat(
    [df.assign(source_split=split_name) for split_name, df in analysis_splits.items()],
    ignore_index=True,
)
global_frames = {"global": global_df}

label_cols = [column for column in train_df.columns if column not in ID_COLUMNS]

print(f"Loaded {len(datasets)} files")
print(f"Detected {len(label_cols)} labels")
print(f"Global analysis rows: {len(global_df):,}")
label_cols


## Reusable helpers

These helpers keep the notebook exploratory without repeating summary and annotation code for every split.


In [ ]:
FRONTAL_VIEWS = {"AP", "PA"}
LATERAL_VIEWS = {"LATERAL", "LL", "LPO", "RPO"}
MAJOR_VIEWS = ["AP", "PA", "LATERAL"]


def format_count_pct(count: int | float, pct: int | float) -> str:
    if pd.isna(count) or pd.isna(pct):
        return ""
    return f"{int(round(count)):,}\n({pct:.1f}%)"


def annotate_bar_containers(ax, labels_by_container: list[list[str]], orientation: str) -> None:
    for container, labels in zip(ax.containers, labels_by_container):
        for bar, label in zip(container.patches, labels):
            if bar is None or not label:
                continue
            if orientation == "horizontal":
                label = label.replace("\n", " ")
                x = bar.get_x() + bar.get_width()
                y = bar.get_y() + bar.get_height() / 2
                ax.annotate(label, (x, y), xytext=(3, 0), textcoords="offset points", ha="left", va="center", fontsize=8)
            else:
                x = bar.get_x() + bar.get_width() / 2
                y = bar.get_y() + bar.get_height()
                ax.annotate(label, (x, y), xytext=(0, 3), textcoords="offset points", ha="center", va="bottom", fontsize=8)
    if orientation == "horizontal":
        ax.margins(x=0.18)
    else:
        ax.margins(y=0.18)


def labels_by_hue(
    plot_df: pd.DataFrame,
    category_col: str,
    hue_col: str,
    count_col: str,
    pct_col: str,
    category_order,
    hue_order,
) -> list[list[str]]:
    keyed = plot_df.set_index([hue_col, category_col])
    labels = []
    for hue_value in hue_order:
        hue_labels = []
        for category_value in category_order:
            key = (hue_value, category_value)
            if key in keyed.index:
                row = keyed.loc[key]
                if isinstance(row, pd.DataFrame):
                    row = row.iloc[0]
                hue_labels.append(format_count_pct(row[count_col], row[pct_col]))
            else:
                hue_labels.append("")
        labels.append(hue_labels)
    return labels


def labels_for_categories(
    plot_df: pd.DataFrame,
    category_col: str,
    count_col: str,
    pct_col: str,
    category_order,
) -> list[str]:
    keyed = plot_df.set_index(category_col)
    labels = []
    for category_value in category_order:
        if category_value in keyed.index:
            row = keyed.loc[category_value]
            if isinstance(row, pd.DataFrame):
                row = row.iloc[0]
            labels.append(format_count_pct(row[count_col], row[pct_col]))
        else:
            labels.append("")
    return labels


def normalize_view_position(series: pd.Series) -> pd.Series:
    return series.fillna("Missing").astype(str).str.strip().str.upper().replace({"": "Missing"})


def format_view_combo(views: tuple[str, ...]) -> str:
    return " + ".join(views) if views else "Missing"


def has_any_view(views: tuple[str, ...], candidates: set[str]) -> bool:
    return bool(set(views) & candidates)


def parse_mimic_study_datetime(df: pd.DataFrame) -> pd.Series:
    if "StudyDate" not in df.columns or "StudyTime" not in df.columns:
        return pd.Series(pd.NaT, index=df.index)

    date_text = (
        df["StudyDate"]
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(8)
    )
    time_text = (
        df["StudyTime"]
        .astype("string")
        .str.replace(r"\.0$", "", regex=True)
        .str.split(".")
        .str[0]
        .str.zfill(6)
        .str[:6]
    )
    return pd.to_datetime(date_text + time_text, format="%Y%m%d%H%M%S", errors="coerce")


def preview_dataset(name: str, df: pd.DataFrame, head_rows: int = 3) -> None:
    print(f"\n{name}")
    print(f"shape: {df.shape}")
    print(f"duplicate dicom_id rows: {df['dicom_id'].duplicated().sum()}")
    display(df.head(head_rows))

    missing = (
        df.isna().sum()
        .rename("missing_count")
        .loc[lambda series: series > 0]
        .sort_values(ascending=False)
    )
    if missing.empty:
        print("No missing values detected.")
    else:
        missing_df = missing.to_frame().assign(
            missing_pct=lambda frame: frame["missing_count"] / len(df) * 100
        )
        display(missing_df)


def build_split_overview(split_frames: dict[str, pd.DataFrame], label_columns: list[str]) -> pd.DataFrame:
    rows = []
    for split_name, df in split_frames.items():
        positive_labels_per_image = df[label_columns].sum(axis=1)
        rows.append(
            {
                "split": split_name,
                "rows": len(df),
                "unique_subjects": df["subject_id"].nunique(),
                "unique_studies": df["study_id"].nunique(),
                "unique_dicoms": df["dicom_id"].nunique(),
                "missing_ViewPosition": df["ViewPosition"].isna().sum(),
                "missing_ViewCodeMeaning": df["ViewCodeSequence_CodeMeaning"].isna().sum(),
                "avg_labels_per_image": positive_labels_per_image.mean(),
                "median_labels_per_image": positive_labels_per_image.median(),
                "no_finding_count": int(df["No Finding"].sum()),
                "no_finding_rate_pct": df["No Finding"].mean() * 100,
            }
        )

    return pd.DataFrame(rows).sort_values("rows", ascending=False).reset_index(drop=True)


def summarize_labels(df: pd.DataFrame, split_name: str, label_columns: list[str]) -> pd.DataFrame:
    summary = (
        df[label_columns]
        .mean()
        .mul(100)
        .rename("positive_rate_pct")
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={"index": "label"})
    )
    summary.insert(0, "split", split_name)
    summary["positive_count"] = summary["label"].map(df[label_columns].sum().to_dict()).astype(int)
    return summary[["split", "label", "positive_count", "positive_rate_pct"]]


def plot_label_prevalence(
    split_frames: dict[str, pd.DataFrame],
    label_columns: list[str],
    reference_df: pd.DataFrame,
    top_n: int = 15,
) -> None:
    label_summary = pd.concat(
        [summarize_labels(df, split_name, label_columns) for split_name, df in split_frames.items()],
        ignore_index=True,
    )
    reference_summary = summarize_labels(reference_df, "global", label_columns)
    label_order = reference_summary.head(top_n)["label"].tolist()
    hue_order = list(split_frames)
    plot_df = label_summary[label_summary["label"].isin(label_order)].copy()

    plt.figure(figsize=(12, max(7, top_n * 0.95)))
    ax = sns.barplot(
        data=plot_df,
        x="positive_rate_pct",
        y="label",
        hue="split",
        order=label_order,
        hue_order=hue_order,
    )
    annotate_bar_containers(
        ax,
        labels_by_hue(plot_df, "label", "split", "positive_count", "positive_rate_pct", label_order, hue_order),
        orientation="horizontal",
    )
    plt.title(f"Top {top_n} labels by global prevalence")
    plt.xlabel("Positive rate (%)")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()


def plot_global_label_prevalence(df: pd.DataFrame, label_columns: list[str], top_n: int = 26) -> None:
    plot_df = summarize_labels(df, "global", label_columns).head(top_n)
    label_order = plot_df["label"].tolist()

    plt.figure(figsize=(10, max(7, top_n * 0.32)))
    ax = sns.barplot(data=plot_df, x="positive_rate_pct", y="label", order=label_order, color="C0")
    annotate_bar_containers(
        ax,
        [labels_for_categories(plot_df, "label", "positive_count", "positive_rate_pct", label_order)],
        orientation="horizontal",
    )
    plt.title(f"Global top {top_n} labels")
    plt.xlabel("Positive rate (%)")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()


def plot_labels_per_image(split_frames: dict[str, pd.DataFrame], label_columns: list[str], include_global: bool = True) -> None:
    frames = dict(split_frames)
    if include_global:
        frames["global"] = pd.concat(split_frames.values(), ignore_index=True)

    fig, axes = plt.subplots(1, len(frames), figsize=(5 * len(frames), 4.5), sharey=True)
    axes = np.atleast_1d(axes)

    for ax, (split_name, df) in zip(axes, frames.items()):
        label_counts = df[label_columns].sum(axis=1).astype(int)
        counts = label_counts.value_counts().sort_index()
        pct = counts / len(df) * 100
        bars = ax.bar(counts.index.astype(str), counts.values)
        ax.bar_label(bars, labels=[format_count_pct(count, pct_value) for count, pct_value in zip(counts.values, pct.values)], padding=3, fontsize=8)
        ax.margins(y=0.18)
        ax.set_title(split_name)
        ax.set_xlabel("Positive labels per image")
        ax.set_ylabel("Image count")

    plt.tight_layout()
    plt.show()


def summarize_view_positions(df: pd.DataFrame, split_name: str, top_n: int = 8) -> pd.DataFrame:
    view_counts = (
        df["ViewPosition"]
        .fillna("Missing")
        .value_counts()
        .head(top_n)
        .rename_axis("ViewPosition")
        .reset_index(name="count")
    )
    view_counts.insert(0, "split", split_name)
    view_counts["rate_pct"] = view_counts["count"] / len(df) * 100
    return view_counts


def plot_view_positions(
    split_frames: dict[str, pd.DataFrame],
    reference_df: pd.DataFrame,
    top_n: int = 6,
) -> None:
    view_order = summarize_view_positions(reference_df, "global", top_n=top_n)["ViewPosition"].tolist()
    plot_df = pd.concat(
        [summarize_view_positions(df, split_name, top_n=999) for split_name, df in split_frames.items()],
        ignore_index=True,
    )
    plot_df = plot_df[plot_df["ViewPosition"].isin(view_order)].copy()
    hue_order = list(split_frames)

    plt.figure(figsize=(12, 5.5))
    ax = sns.barplot(
        data=plot_df,
        x="ViewPosition",
        y="rate_pct",
        hue="split",
        order=view_order,
        hue_order=hue_order,
    )
    annotate_bar_containers(
        ax,
        labels_by_hue(plot_df, "ViewPosition", "split", "count", "rate_pct", view_order, hue_order),
        orientation="vertical",
    )
    plt.title(f"Top {top_n} view positions by global frequency")
    plt.xlabel("")
    plt.ylabel("Rate (%)")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


def plot_global_view_positions(df: pd.DataFrame, top_n: int = 8) -> None:
    plot_df = summarize_view_positions(df, "global", top_n=top_n)
    view_order = plot_df["ViewPosition"].tolist()

    plt.figure(figsize=(10, 5))
    ax = sns.barplot(data=plot_df, x="ViewPosition", y="rate_pct", order=view_order, color="C0")
    annotate_bar_containers(
        ax,
        [labels_for_categories(plot_df, "ViewPosition", "count", "rate_pct", view_order)],
        orientation="vertical",
    )
    plt.title(f"Global top {top_n} view positions")
    plt.xlabel("")
    plt.ylabel("Rate (%)")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


def compute_overlap(split_frames: dict[str, pd.DataFrame], key: str) -> pd.DataFrame:
    split_names = list(split_frames)
    key_sets = {split_name: set(df[key]) for split_name, df in split_frames.items()}

    overlap_matrix = pd.DataFrame(index=split_names, columns=split_names, dtype=int)
    for left_name in split_names:
        for right_name in split_names:
            overlap_matrix.loc[left_name, right_name] = len(key_sets[left_name] & key_sets[right_name])

    return overlap_matrix


def summarize_no_finding_conflicts(split_frames: dict[str, pd.DataFrame], label_columns: list[str]) -> pd.DataFrame:
    other_labels = [label for label in label_columns if label != "No Finding"]
    rows = []

    for split_name, df in split_frames.items():
        no_finding_count = int(df["No Finding"].sum())
        conflicts = int(((df["No Finding"] == 1) & (df[other_labels].sum(axis=1) > 0)).sum())
        rows.append(
            {
                "split": split_name,
                "rows_with_no_finding": no_finding_count,
                "rows_with_no_finding_and_other_label": conflicts,
                "conflict_rate_within_no_finding_pct": conflicts / max(no_finding_count, 1) * 100,
            }
        )

    return pd.DataFrame(rows)


def summarize_submission_file(df: pd.DataFrame, split_name: str, label_columns: list[str]) -> pd.DataFrame:
    submission_label_columns = [column for column in df.columns if column != "dicom_id"]
    rows = [
        {
            "split": split_name,
            "rows": len(df),
            "label_columns_match_train": submission_label_columns == label_columns,
            "min_score": df[submission_label_columns].min().min(),
            "mean_score": df[submission_label_columns].mean().mean(),
            "max_score": df[submission_label_columns].max().max(),
        }
    ]
    return pd.DataFrame(rows)


In [ ]:
def build_study_view_table(df: pd.DataFrame, label_columns: list[str]) -> pd.DataFrame:
    temp = df.assign(_view=normalize_view_position(df["ViewPosition"]))
    group_keys = ["subject_id", "study_id"]

    counts = (
        temp.groupby(group_keys, as_index=False)
        .agg(
            image_count=("dicom_id", "size"),
            unique_dicoms=("dicom_id", "nunique"),
        )
    )
    views = (
        temp.groupby(group_keys)["_view"]
        .agg(lambda values: tuple(sorted(set(values))))
        .rename("views")
        .reset_index()
    )
    study_labels = (
        temp.groupby(group_keys)[label_columns]
        .max()
        .sum(axis=1)
        .rename("positive_labels_per_study")
        .reset_index()
    )

    study_table = counts.merge(views, on=group_keys).merge(study_labels, on=group_keys)
    study_table["view_combo"] = study_table["views"].map(format_view_combo)
    study_table["has_frontal"] = study_table["views"].map(lambda views: has_any_view(views, FRONTAL_VIEWS))
    study_table["has_lateral"] = study_table["views"].map(lambda views: has_any_view(views, LATERAL_VIEWS))
    study_table["has_pa"] = study_table["views"].map(lambda views: "PA" in views)
    study_table["has_ap"] = study_table["views"].map(lambda views: "AP" in views)
    study_table["has_frontal_and_lateral"] = study_table["has_frontal"] & study_table["has_lateral"]
    return study_table


def summarize_study_views(split_frames: dict[str, pd.DataFrame], label_columns: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    tables = []
    summary_rows = []

    for split_name, df in split_frames.items():
        study_table = build_study_view_table(df, label_columns).assign(split=split_name)
        tables.append(study_table)
        summary_rows.append(
            {
                "split": split_name,
                "studies": len(study_table),
                "single_image_studies_count": int((study_table["image_count"] == 1).sum()),
                "single_image_studies_pct": (study_table["image_count"] == 1).mean() * 100,
                "median_images_per_study": study_table["image_count"].median(),
                "frontal_and_lateral_studies_count": int(study_table["has_frontal_and_lateral"].sum()),
                "frontal_and_lateral_studies_pct": study_table["has_frontal_and_lateral"].mean() * 100,
                "pa_and_lateral_studies_pct": (study_table["has_pa"] & study_table["has_lateral"]).mean() * 100,
                "ap_and_lateral_studies_pct": (study_table["has_ap"] & study_table["has_lateral"]).mean() * 100,
                "median_positive_labels_per_study": study_table["positive_labels_per_study"].median(),
            }
        )

    all_studies = pd.concat(tables, ignore_index=True)
    summary = pd.DataFrame(summary_rows)
    return all_studies, summary


def summarize_view_combos(study_table: pd.DataFrame, scope_col: str = "split", top_order=None) -> pd.DataFrame:
    combo_counts = (
        study_table.groupby([scope_col, "view_combo"])
        .size()
        .rename("study_count")
        .reset_index()
    )
    totals = study_table.groupby(scope_col).size().rename("total_studies")
    combo_counts = combo_counts.merge(totals, on=scope_col)
    combo_counts["study_rate_pct"] = combo_counts["study_count"] / combo_counts["total_studies"] * 100
    if top_order is not None:
        combo_counts = combo_counts[combo_counts["view_combo"].isin(top_order)].copy()
    return combo_counts


def plot_top_view_combos(study_table: pd.DataFrame, top_n: int = 12) -> None:
    combo_order = (
        study_table.groupby("view_combo")
        .size()
        .sort_values(ascending=False)
        .head(top_n)
        .index
        .tolist()
    )
    plot_df = summarize_view_combos(study_table, top_order=combo_order)
    hue_order = [split_name for split_name in analysis_splits if split_name in plot_df["split"].unique()]

    plt.figure(figsize=(13, max(5, top_n * 0.35)))
    ax = sns.barplot(
        data=plot_df,
        x="study_count",
        y="view_combo",
        hue="split",
        order=combo_order,
        hue_order=hue_order,
    )
    annotate_bar_containers(
        ax,
        labels_by_hue(plot_df, "view_combo", "split", "study_count", "study_rate_pct", combo_order, hue_order),
        orientation="horizontal",
    )
    plt.title(f"Top {top_n} study view combinations")
    plt.xlabel("Study count")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()


def plot_global_top_view_combos(study_table: pd.DataFrame, top_n: int = 12) -> None:
    global_table = study_table.drop(columns=["split"], errors="ignore").assign(split="global")
    combo_order = (
        global_table["view_combo"]
        .value_counts()
        .head(top_n)
        .index
        .tolist()
    )
    plot_df = summarize_view_combos(global_table, top_order=combo_order)

    plt.figure(figsize=(11, max(5, top_n * 0.35)))
    ax = sns.barplot(data=plot_df, x="study_count", y="view_combo", order=combo_order, color="C0")
    annotate_bar_containers(
        ax,
        [labels_for_categories(plot_df, "view_combo", "study_count", "study_rate_pct", combo_order)],
        orientation="horizontal",
    )
    plt.title(f"Global top {top_n} study view combinations")
    plt.xlabel("Study count")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()


def build_study_timeline(images_df: pd.DataFrame, label_columns: list[str], scope_columns: list[str] | None = None) -> pd.DataFrame:
    scope_columns = scope_columns or []
    temp = images_df.assign(_view=normalize_view_position(images_df["ViewPosition"]))
    group_keys = scope_columns + ["subject_id", "study_id"]

    base = (
        temp.groupby(group_keys, as_index=False)
        .agg(
            image_count=("dicom_id", "size"),
            study_datetime=("study_datetime", "min"),
            first_dicom_id=("dicom_id", "first"),
        )
    )
    views = (
        temp.groupby(group_keys)["_view"]
        .agg(lambda values: tuple(sorted(set(values))))
        .rename("views")
        .reset_index()
    )
    labels = (
        temp.groupby(group_keys)[label_columns]
        .max()
        .sum(axis=1)
        .rename("positive_labels_per_study")
        .reset_index()
    )

    timeline = base.merge(views, on=group_keys).merge(labels, on=group_keys)
    timeline["view_combo"] = timeline["views"].map(format_view_combo)
    sort_columns = scope_columns + ["subject_id", "study_datetime", "study_id"]
    timeline = timeline.sort_values(sort_columns, na_position="last")
    subject_group = scope_columns + ["subject_id"]
    timeline["days_since_previous_study"] = (
        timeline.groupby(subject_group)["study_datetime"]
        .diff()
        .dt.total_seconds()
        .div(86400)
    )
    return timeline


def summarize_patient_timelines(
    study_timeline: pd.DataFrame,
    scope_columns: list[str] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    scope_columns = scope_columns or []
    patient_group = scope_columns + ["subject_id"]
    patient_summary = (
        study_timeline.groupby(patient_group, as_index=False)
        .agg(
            study_count=("study_id", "nunique"),
            first_study_datetime=("study_datetime", "min"),
            last_study_datetime=("study_datetime", "max"),
            total_images=("image_count", "sum"),
        )
    )
    patient_summary["span_days"] = (
        patient_summary["last_study_datetime"] - patient_summary["first_study_datetime"]
    ).dt.total_seconds().div(86400)

    if scope_columns:
        timeline_summary = (
            patient_summary.groupby(scope_columns)
            .agg(
                patients=("subject_id", "nunique"),
                patients_with_multiple_studies=("study_count", lambda values: int((values > 1).sum())),
                median_studies_per_patient=("study_count", "median"),
                max_studies_per_patient=("study_count", "max"),
                median_span_days=("span_days", "median"),
                max_span_days=("span_days", "max"),
            )
            .reset_index()
        )
    else:
        timeline_summary = pd.DataFrame(
            [
                {
                    "scope": "global",
                    "patients": patient_summary["subject_id"].nunique(),
                    "patients_with_multiple_studies": int((patient_summary["study_count"] > 1).sum()),
                    "median_studies_per_patient": patient_summary["study_count"].median(),
                    "max_studies_per_patient": patient_summary["study_count"].max(),
                    "median_span_days": patient_summary["span_days"].median(),
                    "max_span_days": patient_summary["span_days"].max(),
                }
            ]
        )

    timeline_summary["multiple_study_patient_pct"] = (
        timeline_summary["patients_with_multiple_studies"] / timeline_summary["patients"] * 100
    )
    return patient_summary, timeline_summary


def label_pair_correlation_table(df: pd.DataFrame, label_columns: list[str]) -> pd.DataFrame:
    corr = df[label_columns].corr()
    pairs = []
    for left_idx, left_label in enumerate(label_columns):
        for right_label in label_columns[left_idx + 1 :]:
            pairs.append(
                {
                    "left_label": left_label,
                    "right_label": right_label,
                    "correlation": corr.loc[left_label, right_label],
                    "cooccurrence_count": int(((df[left_label] == 1) & (df[right_label] == 1)).sum()),
                    "left_positive_count": int(df[left_label].sum()),
                    "right_positive_count": int(df[right_label].sum()),
                }
            )
    return pd.DataFrame(pairs).dropna(subset=["correlation"])


def view_conditioned_prevalence(
    split_frames: dict[str, pd.DataFrame],
    label_columns: list[str],
    major_views: list[str] = MAJOR_VIEWS,
) -> pd.DataFrame:
    rows = []
    for split_name, df in split_frames.items():
        temp = df.assign(_view=normalize_view_position(df["ViewPosition"]))
        for view_name, view_df in temp[temp["_view"].isin(major_views)].groupby("_view"):
            for label in label_columns:
                positives = int(view_df[label].sum())
                rows.append(
                    {
                        "split": split_name,
                        "ViewPosition": view_name,
                        "label": label,
                        "row_count": len(view_df),
                        "positive_count": positives,
                        "positive_rate_pct": positives / len(view_df) * 100 if len(view_df) else np.nan,
                    }
                )
    return pd.DataFrame(rows)


def plot_view_conditioned_prevalence(prevalence_df: pd.DataFrame, label_order: list[str], title: str) -> None:
    plot_df = prevalence_df[prevalence_df["label"].isin(label_order)].copy()
    hue_order = [view for view in MAJOR_VIEWS if view in plot_df["ViewPosition"].unique()]

    plt.figure(figsize=(12, max(6, len(label_order) * 0.45)))
    ax = sns.barplot(
        data=plot_df,
        x="positive_rate_pct",
        y="label",
        hue="ViewPosition",
        order=label_order,
        hue_order=hue_order,
    )
    annotate_bar_containers(
        ax,
        labels_by_hue(plot_df, "label", "ViewPosition", "positive_count", "positive_rate_pct", label_order, hue_order),
        orientation="horizontal",
    )
    plt.title(title)
    plt.xlabel("Positive rate (%)")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()


def resolve_relative_path(root: Path, path_value: str) -> Path:
    path = Path(path_value)
    return path if path.is_absolute() else root / path


def summarize_image_paths(split_frames: dict[str, pd.DataFrame], image_root: Path) -> pd.DataFrame:
    rows = []
    for split_name, df in split_frames.items():
        paths = df["path"].dropna().astype(str)
        exists_flags = paths.map(lambda value: resolve_relative_path(image_root, value).is_file())
        rows.append(
            {
                "split": split_name,
                "rows": len(df),
                "path_values": len(paths),
                "unique_paths": paths.nunique(),
                "duplicate_path_rows": int(paths.duplicated().sum()),
                "jpg_extension_rows": int(paths.str.lower().str.endswith(".jpg").sum()),
                "existing_files": int(exists_flags.sum()),
                "missing_files": int((~exists_flags).sum()),
                "existing_file_pct": exists_flags.mean() * 100 if len(exists_flags) else np.nan,
            }
        )
    return pd.DataFrame(rows)


def read_report_text(report_path: str, max_chars: int = 3000) -> str | None:
    if not report_path:
        return None
    full_path = mimic_cxr_dir / report_path
    if not full_path.is_file():
        return None
    return full_path.read_text(errors="replace")[:max_chars]


## 1) Quick Split Overview

Start here to answer: how big are the splits, what fields exist, and where is metadata incomplete?


In [ ]:
for split_name, df in analysis_splits.items():
    preview_dataset(split_name, df)

split_overview_df = build_split_overview(analysis_splits, label_cols)
display(split_overview_df)


## 2) Global Dataset Statistics

These summaries disregard train/development/test and treat the CXR-LT 2023 labeled rows as one dataset.


In [ ]:
global_overview_df = build_split_overview(global_frames, label_cols).rename(columns={"split": "scope"})
global_label_summary_df = summarize_labels(global_df, "global", label_cols)
global_view_position_summary_df = summarize_view_positions(global_df, "global", top_n=12).rename(columns={"split": "scope"})
global_no_finding_conflicts_df = summarize_no_finding_conflicts(global_frames, label_cols).rename(columns={"split": "scope"})

display(global_overview_df)

print("Most common global labels")
display(global_label_summary_df.head(10))

print("Rarest global labels")
display(global_label_summary_df.tail(10))

print("Global view-position distribution")
display(global_view_position_summary_df)

print("Global No Finding conflicts")
display(global_no_finding_conflicts_df)

plot_global_label_prevalence(global_df, label_cols, top_n=26)
plot_global_view_positions(global_df, top_n=8)


## 3) Label Imbalance And Multi-Label Density

This is the part that tells you why the task is called long-tailed. Pay attention to the rarest labels and to how many findings co-occur on each image.


In [ ]:
label_summary_df = pd.concat(
    [summarize_labels(df, split_name, label_cols) for split_name, df in analysis_splits.items()],
    ignore_index=True,
)
label_summary_with_global_df = pd.concat([label_summary_df, global_label_summary_df], ignore_index=True)

display(label_summary_with_global_df.query("split == 'global'").head(10))
display(label_summary_with_global_df.query("split == 'global'").tail(10))

display(label_summary_df.query("split == 'train'").head(10))
display(label_summary_df.query("split == 'train'").tail(10))

plot_label_prevalence(analysis_splits, label_cols, reference_df=global_df, top_n=26)
plot_labels_per_image(analysis_splits, label_cols, include_global=True)

display(summarize_no_finding_conflicts(analysis_splits, label_cols))


## 4) View Metadata And Split Leakage Checks

For modeling, it helps to know whether view-position distribution shifts across splits and whether the same patient or study appears in multiple splits.


In [ ]:
view_position_summary_df = pd.concat(
    [summarize_view_positions(df, split_name) for split_name, df in analysis_splits.items()],
    ignore_index=True,
)
view_position_summary_with_global_df = pd.concat(
    [view_position_summary_df, global_view_position_summary_df.rename(columns={"scope": "split"})],
    ignore_index=True,
)

display(view_position_summary_with_global_df)
plot_view_positions(analysis_splits, reference_df=global_df, top_n=6)

for overlap_key in ["subject_id", "study_id", "dicom_id"]:
    print(f"\nOverlap for {overlap_key}")
    display(compute_overlap(analysis_splits, overlap_key))


## 5) Sample Submission Sanity Checks

These files are handy for confirming the label order expected by the benchmark and for checking score ranges before writing an inference pipeline.


In [ ]:
submission_summary_df = pd.concat(
    [summarize_submission_file(df, split_name, label_cols) for split_name, df in submission_splits.items()],
    ignore_index=True,
)
display(submission_summary_df)

for split_name, df in submission_splits.items():
    print(f"\n{split_name}")
    display(df.head(3))


## 6) Study-Level Aggregation

This checks whether a `study_id` usually has a frontal view, a lateral view, or both. This matters for multi-view fusion because the image-level rows are not the same unit as a clinical exam.


In [ ]:
study_table_df, study_view_summary_df = summarize_study_views(analysis_splits, label_cols)
global_study_table_df, global_study_view_summary_df = summarize_study_views(global_frames, label_cols)
global_study_view_summary_df = global_study_view_summary_df.rename(columns={"split": "scope"})

display(study_view_summary_df)
display(global_study_view_summary_df)

display(
    global_study_table_df
    [
        [
            "subject_id",
            "study_id",
            "image_count",
            "view_combo",
            "has_frontal_and_lateral",
            "positive_labels_per_study",
        ]
    ]
    .head(10)
)

plot_top_view_combos(study_table_df, top_n=12)
plot_global_top_view_combos(global_study_table_df, top_n=12)


## 7) Patient Timeline Behavior

The CXR-LT CSVs do not include acquisition time, so this joins MIMIC-CXR-JPG metadata when available. MIMIC dates are shifted, but within-patient ordering and intervals are still useful for repeat-exam analysis.


In [ ]:
metadata_df, metadata_path = load_first_existing(
    [
        mimic_cxr_jpg_dir / "mimic-cxr-2.0.0-metadata.csv.gz",
        mimic_cxr_jpg_dir / "mimic-cxr-2.0.0-metadata.csv",
    ]
)

if metadata_df is None:
    print("MIMIC-CXR-JPG metadata was not found. Timeline cells will fall back to study_id ordering.")
    metadata_columns = ["dicom_id", "study_datetime"]
    metadata_for_join = pd.DataFrame(columns=metadata_columns)
else:
    print(f"Loaded metadata: {metadata_path}")
    metadata_df["study_datetime"] = parse_mimic_study_datetime(metadata_df)
    metadata_columns = [
        column
        for column in [
            "dicom_id",
            "StudyDate",
            "StudyTime",
            "study_datetime",
            "PerformedProcedureStepDescription",
            "ProcedureCodeSequence_CodeMeaning",
            "Rows",
            "Columns",
        ]
        if column in metadata_df.columns
    ]
    metadata_for_join = metadata_df[metadata_columns].copy()

all_labeled_images_df = global_df.merge(metadata_for_join, on="dicom_id", how="left")

print(f"Rows with parsed study_datetime: {all_labeled_images_df['study_datetime'].notna().sum():,} / {len(all_labeled_images_df):,}")
display(all_labeled_images_df.head(3))


In [ ]:
study_timeline_df = build_study_timeline(all_labeled_images_df, label_cols, scope_columns=["source_split"])
global_study_timeline_df = build_study_timeline(all_labeled_images_df, label_cols)

patient_timeline_summary_df, split_patient_summary_df = summarize_patient_timelines(
    study_timeline_df,
    scope_columns=["source_split"],
)
global_patient_timeline_summary_df, global_patient_summary_df = summarize_patient_timelines(global_study_timeline_df)

split_patient_summary_df = split_patient_summary_df.rename(columns={"source_split": "split"})

display(split_patient_summary_df)
display(global_patient_summary_df)

display(
    global_patient_timeline_summary_df
    .sort_values(["study_count", "span_days"], ascending=[False, False])
    .head(10)
)


In [ ]:
top_patient_row = (
    global_patient_timeline_summary_df
    .sort_values(["study_count", "span_days"], ascending=[False, False])
    .head(1)
)

if top_patient_row.empty:
    print("No patient timeline rows available.")
else:
    top_subject_id = top_patient_row.iloc[0]["subject_id"]
    print(f"Example repeat-exam patient: subject_id={top_subject_id}")
    display(
        global_study_timeline_df
        .query("subject_id == @top_subject_id")
        [
            [
                "subject_id",
                "study_id",
                "study_datetime",
                "days_since_previous_study",
                "image_count",
                "view_combo",
                "positive_labels_per_study",
            ]
        ]
    )


## 8) Global Label Co-Occurrence And Correlations

These checks disregard split and use the complete labeled CXR-LT 2023 dataset. The correlation heatmap uses Pearson correlation on binary labels, which is the phi coefficient for two binary variables.


In [ ]:
global_corr_df = global_df[label_cols].corr()
pair_corr_df = label_pair_correlation_table(global_df, label_cols)

plt.figure(figsize=(14, 12))
sns.heatmap(global_corr_df, cmap="vlag", center=0, vmin=-1, vmax=1, square=True, linewidths=0.2)
plt.title("Global label correlation matrix")
plt.tight_layout()
plt.show()

print("Most positively correlated label pairs")
display(pair_corr_df.sort_values("correlation", ascending=False).head(15))

print("Most negatively correlated label pairs")
display(pair_corr_df.sort_values("correlation", ascending=True).head(15))


In [ ]:
global_cooccurrence_df = global_df[label_cols].T.dot(global_df[label_cols])
label_order = global_df[label_cols].sum().sort_values(ascending=False).index

plt.figure(figsize=(14, 12))
sns.heatmap(
    global_cooccurrence_df.loc[label_order, label_order],
    cmap="mako",
    square=True,
    linewidths=0.2,
)
plt.title("Global label co-occurrence counts")
plt.tight_layout()
plt.show()

display(global_cooccurrence_df.loc[label_order, label_order].iloc[:10, :10])


## 9) View-Conditioned Prevalence

This compares label rates for `AP`, `PA`, and `LATERAL`. It is useful for checking whether a label is view-dependent enough that frontal and lateral encoders should be treated separately.


In [ ]:
view_prevalence_df = view_conditioned_prevalence(analysis_splits, label_cols)
global_view_prevalence_df = view_conditioned_prevalence(global_frames, label_cols)

view_counts_df = (
    view_prevalence_df[["split", "ViewPosition", "row_count"]]
    .drop_duplicates()
    .sort_values(["split", "ViewPosition"])
)
global_view_counts_df = (
    global_view_prevalence_df[["split", "ViewPosition", "row_count"]]
    .drop_duplicates()
    .rename(columns={"split": "scope"})
    .sort_values(["scope", "ViewPosition"])
)

display(view_counts_df)
display(global_view_counts_df)

top_global_labels = global_df[label_cols].sum().sort_values(ascending=False).head(12).index.tolist()
plot_view_conditioned_prevalence(
    global_view_prevalence_df,
    label_order=top_global_labels,
    title="Global label prevalence by view position",
)

display(
    global_view_prevalence_df
    .query("label in @top_global_labels")
    .pivot(index="label", columns="ViewPosition", values="positive_rate_pct")
    .loc[top_global_labels]
    .round(3)
)


## 10) Image Availability And Path Sanity

CXR-LT `path` values are relative to the MIMIC-CXR-JPG root. This checks whether the expected JPG files are present locally before dataloader work.


In [ ]:
image_root = mimic_cxr_jpg_dir
print(f"image_root: {image_root}")
print(f"image_root exists: {image_root.exists()}")
print(f"image files directory exists: {(image_root / 'files').exists()}")

image_path_summary_df = summarize_image_paths(analysis_splits, image_root)
global_image_path_summary_df = summarize_image_paths(global_frames, image_root).rename(columns={"split": "scope"})

display(image_path_summary_df)
display(global_image_path_summary_df)

missing_global_paths = global_df.loc[
    ~global_df["path"].astype(str).map(lambda value: resolve_relative_path(image_root, value).is_file()),
    ["source_split", "dicom_id", "subject_id", "study_id", "ViewPosition", "path"],
].head(10)

print("Sample missing global image paths")
display(missing_global_paths)


## 11) Text Linkage Readiness

This joins CXR-LT studies to the MIMIC-CXR study list. A successful index join means the label row can be paired to a report path. File existence is a separate check because this workstation may have metadata without the report text tree.


In [ ]:
study_list_df, study_list_path = load_first_existing(
    [
        mimic_cxr_dir / "cxr-study-list.csv.gz",
        mimic_cxr_dir / "cxr-study-list.csv",
    ]
)

if study_list_df is None:
    print("MIMIC-CXR study list was not found. Report linkage cannot be checked.")
    report_link_summary_df = pd.DataFrame()
    global_report_link_summary_df = pd.DataFrame()
    linked_reports_df = pd.DataFrame()
    global_linked_reports_df = pd.DataFrame()
else:
    print(f"Loaded study list: {study_list_path}")
    study_list_for_join = study_list_df.rename(columns={"path": "report_path"})
    unique_studies_df = pd.concat(
        [
            df[["subject_id", "study_id"]].drop_duplicates().assign(split=split_name)
            for split_name, df in analysis_splits.items()
        ],
        ignore_index=True,
    )
    global_unique_studies_df = global_df[["subject_id", "study_id"]].drop_duplicates().assign(scope="global")

    linked_reports_df = unique_studies_df.merge(
        study_list_for_join[["subject_id", "study_id", "report_path"]],
        on=["subject_id", "study_id"],
        how="left",
    )
    global_linked_reports_df = global_unique_studies_df.merge(
        study_list_for_join[["subject_id", "study_id", "report_path"]],
        on=["subject_id", "study_id"],
        how="left",
    )

    for linked_df in [linked_reports_df, global_linked_reports_df]:
        linked_df["has_report_index"] = linked_df["report_path"].notna()
        linked_df["report_file_exists"] = linked_df["report_path"].fillna("").map(
            lambda value: (mimic_cxr_dir / value).is_file() if value else False
        )

    report_link_summary_df = (
        linked_reports_df.groupby("split")
        .agg(
            studies=("study_id", "nunique"),
            studies_with_report_index=("has_report_index", "sum"),
            studies_with_report_file=("report_file_exists", "sum"),
        )
        .reset_index()
    )
    report_link_summary_df["report_index_pct"] = (
        report_link_summary_df["studies_with_report_index"] / report_link_summary_df["studies"] * 100
    )
    report_link_summary_df["report_file_pct"] = (
        report_link_summary_df["studies_with_report_file"] / report_link_summary_df["studies"] * 100
    )

    global_report_link_summary_df = (
        global_linked_reports_df.groupby("scope")
        .agg(
            studies=("study_id", "nunique"),
            studies_with_report_index=("has_report_index", "sum"),
            studies_with_report_file=("report_file_exists", "sum"),
        )
        .reset_index()
    )
    global_report_link_summary_df["report_index_pct"] = (
        global_report_link_summary_df["studies_with_report_index"] / global_report_link_summary_df["studies"] * 100
    )
    global_report_link_summary_df["report_file_pct"] = (
        global_report_link_summary_df["studies_with_report_file"] / global_report_link_summary_df["studies"] * 100
    )

    print(f"report files directory exists: {(mimic_cxr_dir / 'files').exists()}")
    display(report_link_summary_df)
    display(global_report_link_summary_df)
    display(linked_reports_df.head(10))


In [ ]:
if global_linked_reports_df.empty or not global_linked_reports_df["report_file_exists"].any():
    print("No local report text files were found to preview.")
else:
    sample_report_row = global_linked_reports_df.query("report_file_exists").iloc[0]
    print(
        "Sample report: "
        f"subject_id={sample_report_row['subject_id']}, "
        f"study_id={sample_report_row['study_id']}"
    )
    print(read_report_text(sample_report_row["report_path"]))


## Notes For The Next Modeling Pass

Use these outputs to decide whether the modular dataloader should preserve study-level grouping, whether paired frontal/lateral samples are common enough to require special batching, and whether the local machine has the image/report files required for multimodal experiments.
